# `operator_gb` — an interactive tutorial

**NOTE:** This notebook can be run online, without installing anything, via Binder:

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ClemensHofstadler/operator_gb/HEAD?labpath=Tutorial.ipynb)

The Binder session comes with SageMath and a preinstalled version of the package `operator_gb`, so all cells below can be executed and modified directly in the browser.

Cells are executed with `Shift`+`Enter`. Let us start by loading the package.

In [1]:
from operator_gb import *

# The Moore-Penrose inverse

Recall that the Moore-Penrose inverse of a complex matrix $A$ (or more generally of a linear operator $A$) is the complex matrix (resp. the linear operator) $B$ satisfying
$$
ABA = A, \qquad BAB = B, \qquad AB = B^\ast A^\ast, \qquad BA = A^\ast B^\ast,
$$
where $P^\ast$ denotes the Hermitian adjoint of a complex matrix (resp. a linear operator) $P$. If it exists, it is unique, and it is denoted by $A^\dagger$.

## How the package proves such statements

Every operator identity $P = Q$ is translated into the noncommutative polynomial $P - Q$, with one indeterminate for every basic operator. A claimed identity then follows from the assumed identities if the polynomial of the claim lies in the ideal generated by the polynomials of the assumptions.

The command `certify` tries to verify such an ideal membership. If it succeeds, it produces a **cofactor representation**
$$
  f = \sum_{i} p_i \, g_i \, q_i
$$
of the claim $f$ in terms of the assumptions $g_i$. This is a certificate that can be checked independently.

# Part 1: Three worked examples

## Uniqueness of the Moore-Penrose inverse
*(Fact 1 in [Hog13, Sec. I.5.7])*

If $B$ and $C$ both satisfy the Moore-Penrose identities for $A$, then $B = C$.

**Required property:** Encoding the Hermitian adjoint $^\ast$

**Strategy:** Add for every assumed identity $P = Q$ also the corresponding adjoint identity $P^\ast = Q^\ast$, and simplify all operator expressions using the following identities before translating them into polynomials:
$$
(P+Q)^\ast = P^\ast + Q^\ast, \qquad\qquad (PQ)^\ast = Q^\ast P^\ast, \qquad\qquad (P^\ast)^\ast = P.
$$

The adjoint of a basic operator is a separate indeterminate, by convention named `a_adj`. The command `pinv` generates the four Moore-Penrose identities, and `add_adj` adds the adjoint of every assumption.

In [2]:
# first create the FreeAlgebra containing all indeterminates
F.<a, b, c, a_adj, b_adj, c_adj> = FreeAlgebra(QQ)

# assumptions and claim are ordinary SageMath noncommutative polynomials
Pinv_B = [a*b*a - a, b*a*b - b, a*b - b_adj*a_adj, b*a - a_adj*b_adj]
# the command pinv generates the polynomials of the Moore-Penrose identities
# the syntax for this command is pinv(var, name_of_the_MP_inverse, var_adj, name_of_the_MP_adj)
Pinv_C = pinv(a, c, a_adj, c_adj)
# the command add_adj adds the corresponding adjoint identities
assumptions = add_adj(Pinv_B + Pinv_C)
claim = b - c

print("The assumptions are %s.\n" % str(assumptions))
print("The claim is %s.\n" % str(claim))

# call certify
proof = certify(assumptions, claim)

print("The proof is:")
print(pretty_print_proof(proof, assumptions))

# checking the proof
expand_cofactors(proof, assumptions) == b-c

The assumptions are [-a + a*b*a, -b + b*a*b, a*b - b_adj*a_adj, b*a - a_adj*b_adj, -a + a*c*a, -c + c*a*c, -a*c + c_adj*a_adj, -c*a + a_adj*c_adj, -a_adj + a_adj*b_adj*a_adj, -b_adj + b_adj*a_adj*b_adj, -a_adj + a_adj*c_adj*a_adj, -c_adj + c_adj*a_adj*c_adj].

The claim is b - c.

Computing a (partial) Groebner basis and reducing the claims...

Done! Ideal membership of all claims could be verified!
The proof is:
b - c = (-c + c*a*c) + b*c_adj*(-a_adj + a_adj*b_adj*a_adj) + b*a*c*(a*b - b_adj*a_adj) - b*(-a + a*c*a)*b + b*(-a*c + c_adj*a_adj) - b*(-a*c + c_adj*a_adj)*b_adj*a_adj - (-b + b*a*b) + (-c*a + a_adj*c_adj)*b*a*c - (-a_adj + a_adj*c_adj*a_adj)*b_adj*c + c*(-a + a*b*a)*c + (b*a - a_adj*b_adj)*c - a_adj*c_adj*(b*a - a_adj*b_adj)*c


True

## Existence of the Moore-Penrose inverse
*(Fact 1 in [Hog13, Sec. I.5.7])*

Every complex matrix has a Moore-Penrose inverse.

**Required property:** Proving an existential statement

**Strategy:** Construct an explicit expression for the existentially quantified variable.

To do this, the package provides dedicated methods, such as the command `find_equivalent_expression`. Here we start from the fact that, for every matrix $A$, there are matrices $B$ and $C$ with $A = BA^\ast A$ and $A = AA^\ast C$. We introduce a dummy indeterminate $X$ satisfying the Moore-Penrose identities for $A$, and then search for an expression that is equal to $X$ but written in terms of $A$, $B$, $C$ and their adjoints.

In [ ]:
F.<a,b,c,a_adj,b_adj,c_adj,x,x_adj> = FreeAlgebra(QQ)

# in this example, we first have to find an expression for the Moore-Penrose inverse
# we do this using our heuristics
# to this end, we introduce a dummy variable x satisfying the Moore-Penrose equations
# and then search for an expression equivalent to x but in terms of a,b,c and their adjoints
assumptions = add_adj([a - b*a_adj*a, a - a*a_adj*c] + pinv(a, x, a_adj, x_adj))
I = NCIdeal(assumptions)
candidates = I.find_equivalent_expression(x)
# one of the candidates shows that X = A^* C B^*
print("Found candidates for the Moore-Penrose inverse: %s\n" % str(candidates))

# we show that the candidate a_adj*c*b_adj is indeed the Moore-Penrose inverse of a
# by showing that it satisfies the four Moore-Penrose equations
MP_candidate = a_adj * c * b_adj
MP_candidate_adj = adj(MP_candidate)
claim = pinv(a, MP_candidate, a_adj, MP_candidate_adj)
print("The assumptions are %s\n" % str(assumptions))
print("The claim is %s\n" % str(claim))

# call certify
proof = certify(assumptions, claim)

print("The proofs consist of %s steps, respectively.\n" % str(list(map(len,proof))))

## The Moore-Penrose inverse is an involution
*(Fact 10 in [Hog13, Sec. I.5.7])*

$(A^\dagger)^\dagger = A$

**Required property:** Several Moore-Penrose inverses at once

**Strategy:** Every Moore-Penrose inverse occurring in a statement gets its own indeterminate and its own set of Moore-Penrose identities. Here `x` is the Moore-Penrose inverse of `a`, and `y` is the Moore-Penrose inverse of `x`; the claim is `a = y`.

In [3]:
F.<a, x, y, a_adj, x_adj, y_adj> = FreeAlgebra(QQ)

# x is the Moore-Penrose inverse of a, and y is the Moore-Penrose inverse of x
Pinv_x = pinv(a, x, a_adj, x_adj)
Pinv_y = pinv(x, y, x_adj, y_adj)

assumptions = add_adj(Pinv_x + Pinv_y)
claim = a - y

print("The assumptions are %s\n" % str(assumptions))
print("The claim is %s\n" % str(claim))

proof = certify(assumptions, claim)

print("\n", pretty_print_proof(proof, assumptions))

The assumptions are [-a + a*x*a, -x + x*a*x, -a*x + x_adj*a_adj, -x*a + a_adj*x_adj, -x + x*y*x, -y + y*x*y, -x*y + y_adj*x_adj, -y*x + x_adj*y_adj, -a_adj + a_adj*x_adj*a_adj, -x_adj + x_adj*a_adj*x_adj, -x_adj + x_adj*y_adj*x_adj, -y_adj + y_adj*x_adj*y_adj]

The claim is a - y

Computing a (partial) Groebner basis and reducing the claims...

Done! Ideal membership of all claims could be verified!

 a - y = (-y + y*x*y) + a*y_adj*(-x_adj + x_adj*a_adj*x_adj) - a*x*y*(-x*a + a_adj*x_adj) - a*(-x + x*y*x)*a + a*(-x*y + y_adj*x_adj) - a*(-x*y + y_adj*x_adj)*a_adj*x_adj - (-a + a*x*a) + (-y*x + x_adj*y_adj)*a*x*y - (-x_adj + x_adj*y_adj*x_adj)*a_adj*y + y*(-x + x*a*x)*y - (-a*x + x_adj*a_adj)*y + x_adj*y_adj*(-a*x + x_adj*a_adj)*y


# Part 2: Encoding properties of matrices and operators

The examples above only needed adjoints. Most statements about matrices and operators involve further properties — being invertible, having full rank, being real, an inclusion of ranges. All of these can be expressed by polynomial identities, sometimes after introducing auxiliary indeterminates. Below, we collect some useful encodings.

| Property | Encoded by |
| :--- | :--- |
| $A^\ast$, the adjoint of $A$ | a separate indeterminate `a_adj`; adjoin the adjoint of *every* assumption (`add_adj`) |
| $I$ is an identity matrix | `i*i - i`, `i - i_adj`, and `a*i - a`, `i*b - b` for all basic operators |
| $A$ has full row rank | `a*u - i` for a fresh indeterminate `u` and `i` modelling the identitiy matrix |
| $A$ has full column rank | `v*a - i` for a fresh indeterminate `v` and `i` modelling the identitiy matrix |
| $A$ is invertible | `a*a_inv - id2` and `a_inv*a - id1` and `id1`, `id2` modelling identitiy matrices |
| $A$ is real | `a - a_c`, with `a_c`, `a_tr` for the conjugate and the transpose of $A$ |
| $A$ is Hermitian | `a - a_adj` |
| $A$ is normal | `a*a_adj - a_adj*a` |
| $A$ is idempotent | `a*a - a` |
| $A$ has orthonormal columns | `a_adj*a - i` with `i` modelling the identitiy matrix |
| $\operatorname{range}(A) \subseteq \operatorname{range}(B)$ | `a - b*x` for a fresh indeterminate `x` |
| $\ker(A) \subseteq \ker(B)$ | assume `a*x`, claim `b*x`, for a fresh indeterminate `x` |

## Identity matrices

As soon as the identity matrix of a space occurs in a statement, it needs its own indeterminate `i`, together with the identities it satisfies:

- `i*i - i` and `i - i_adj`, since it is an idempotent, Hermitian operator;
- `a*i - a` and `i*b - b` for every basic operator `a`, `b` for which these products are defined.

The second group is easy to forget, and forgetting it is the most common reason why a proof that "should" work does not.

Note that different spaces have different identity matrices: if $A$ maps space 1 to space 2, then $I_1$ and $I_2$ are two distinct indeterminates, with $AI_1 = A = I_2A$.

## One-sided inverses: full rank, injectivity, surjectivity, invertibility

A matrix has full row rank exactly iff it has a right inverse, and full column rank exactly iff it has a left inverse. So full row rank of $A$ is encoded by
$$
AU = I,
$$
where $U$ is a fresh indeterminate satisfying no further hypotheses and $I$ is an identity matrix as in point 3. Analogously, full column rank of $A$ corresponds to $VA = I$, and an invertible $A$ satisfies both with the *same* $U = V$.

More generally, the same trick encodes surjectivity and injectivity of operators. Note that nothing at all is assumed about $U$ and $V$ beyond $AU = I$ (resp. $VA = I$).

## Real matrices

The Hermitian adjoint decomposes into complex conjugation and transposition, $A^\ast = (A^C)^T$, and being real is then expressed by $A = A^C$.

In terms of polynomials, this means introducing two further indeterminates `a_c` and `a_tr` for every basic operator and adding the polynomial `a - a_c`. Additionally, for every assumption $P = Q$, the adjoint identity $P^\ast = Q^\ast$, the transposed identity $P^T = Q^T$, and the conjugated identity $P^C = Q^C$ have to be translated into polynomials as well. These additional identities first have to be simplified using the following rules, which relate the three function symbols to each other. Let $\alpha, \beta, \gamma, \delta \in \{\ast, C, T\}$ with $\gamma \neq \alpha, \beta$ and $\delta \neq C$. Then

- $(P+Q)^\alpha = P^\alpha + Q^\alpha$
- $(P^\alpha)^\beta = P$ if $\alpha = \beta$, and $(P^\alpha)^\beta = P^\gamma$ if $\alpha \neq \beta$
- $(PQ)^C = P^C Q^C$
- $(PQ)^\delta = Q^\delta P^\delta$

## Special classes of matrices

Many properties are already polynomial identities and need no auxiliary indeterminates at all:

| $A$ is ... | polynomial |
| :--- | :--- |
| Hermitian, $A = A^\ast$ | `a - a_adj` |
| normal, $AA^\ast = A^\ast A$ | `a*a_adj - a_adj*a` |
| idempotent, $A^2 = A$ | `a*a - a` |
| an orthogonal projection | `a*a - a` and `a - a_adj` |
| a partial isometry, $A^\dagger = A^\ast$ | `a_dag - a_adj` |
| EP, $AA^\dagger = A^\dagger A$ | `a*a_dag - a_dag*a` |

Together with an identity matrix `i` as in point 3, orthonormal columns of $A$ become `a_adj*a - i` and orthonormal rows become `a*a_adj - i`.

## Ranges and kernels

By a classical result called *Douglas' lemma*, an inclusion of ranges is equivalent to a factorisation:
$$
  \operatorname{range}(P) \subseteq \operatorname{range}(Q) \iff P = QX \text{ for some } X.
$$
So a range inclusion is *assumed* by adding the polynomial `p - q*x` for a fresh indeterminate `x`, and it is *proven* by finding an explicit $X$. The command `find_equivalent_expression` does the latter: `I.find_equivalent_expression(p, prefix=q)` searches the ideal for an element of the form $P - Q(\dots)$.

Kernel inclusions are even simpler. Since
$$
  \ker(P) \subseteq \ker(Q) \iff \bigl(Px = 0 \implies Qx = 0\bigr),
$$
one adds `p*x` to the assumptions, for a fresh indeterminate `x`, and claims `q*x`.

In [4]:
F.<a, a_adj, a_dag, a_dag_adj> = FreeAlgebra(QQ)
I = NCIdeal(add_adj(pinv(a, a_dag, a_adj, a_dag_adj)))

# range(A) is contained in range(A A^dag), because A = A A^dag X for X = A
print(I.find_equivalent_expression(a, prefix=a*a_dag, heuristic='naive')[0])

a - a*a_dag*a


# Part 3: Try yourself

Below are eleven theorems about the Moore-Penrose inverse, roughly in increasing order of difficulty. For each of them the statement is given, but the code is only partly given: everywhere you see `...`, something is missing.

A few reminders:

- `pinv(a, b, a_adj, b_adj)` returns the four Moore-Penrose identities saying that `b` is the Moore-Penrose inverse of `a`, where `a_adj` and `b_adj` are the adjoints of `a` and `b`;
- `add_adj(assumptions)` adds the adjoint of every assumption;
- `certify(assumptions, claim)` reports whether it could verify the claim, and returns the certificate.
- if `certify` fails, look first for a missing assumption — a forgotten identity-matrix relation, or a forgotten adjoint — and only then increase `maxiter`.

## Exercise 1: Hermitian idempotents
*(Fact 13 in [Hog13, Sec. I.5.7])*

If $A = A^\ast$ and $A = A^2$, then $A^\dagger = A$.

*Needed:* adjoints.

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# encode that A is Hermitian and that A is idempotent
a_hermitian = ...
a_idempotent = ...

assumptions = add_adj(Pinv_a + [a_hermitian, a_idempotent])
# state the claim A^dag = A
claim = ...

proof = certify(assumptions, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 2: The Moore-Penrose inverse of the adjoint
*(Fact 10 in [Hog13, Sec. I.5.7])*

$(A^\ast)^\dagger = (A^\dagger)^\ast$

*Needed:* a second set of Moore-Penrose identities, as in the involution example of Part 1. Here `x` denotes $(A^\ast)^\dagger$ and `x_adj` its adjoint. Mind the order of the arguments of `pinv`: the first argument is the operator, the second its Moore-Penrose inverse, and the last two are their adjoints.

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj, x, x_adj> = FreeAlgebra(QQ)

# a_dag is the Moore-Penrose inverse of a
Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)
# x is the Moore-Penrose inverse of a_adj
Pinv_a_adj = pinv(...)

assumptions = add_adj(Pinv_a + Pinv_a_adj)
# state the claim (A^*)^dag = (A^dag)^*
claim = ...

proof = certify(assumptions, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 3: Normal matrices commute with their Moore-Penrose inverse
*(Fact 15 in [Hog13, Sec. I.5.7])*

If $A$ is normal, i.e., $AA^\ast = A^\ast A$, then $AA^\dagger = A^\dagger A$.

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# encode that A is normal
a_normal = ...

assumptions = add_adj(Pinv_a + [a_normal])
# state the claim A A^dag = A^dag A
claim = ...

proof = certify(assumptions, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 4: Orthonormal columns or rows
*(Fact 12 in [Hog13, Sec. I.5.7])*

If $A$ has orthonormal columns or orthonormal rows, then $A^\dagger = A^\ast$.

*Needed:* identity matrices. Here `id1` is the identity on the source of $A$ and `id2` the identity on its target. This is really two statements, so there are two proofs to run — and note that the two cases share all their other assumptions.

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj, id1, id1_adj, id2, id2_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# id1 is the identity on the source of a, id2 the identity on its target;
# the first two relations say A*I1 = A = I2*A -- add the two analogous
# relations for a_dag, which maps in the opposite direction
identity_matrix = [a*id1 - a, id2*a - a, ..., ...]

# encode that A has orthonormal columns, resp. orthonormal rows
a_orthonormal_cols = ...
a_orthonormal_rows = ...

# state the claim A^dag = A^*
claim = ...

# case 1: orthonormal columns
assumptions_1 = add_adj(Pinv_a + identity_matrix + [a_orthonormal_cols])
proof = certify(assumptions_1, claim)
print("The proof consists of %d steps.\n" % len(proof))

# case 2: orthonormal rows
assumptions_2 = add_adj(Pinv_a + identity_matrix + [a_orthonormal_rows])
proof = certify(assumptions_2, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 5: Non-singular square matrices
*(Fact 11 in [Hog13, Sec. I.5.7])*

If $A$ is a non-singular square matrix, then $A^\dagger = A^{-1}$.

*Needed:* identity matrices and a two-sided inverse. All relations for the two identity matrices are already given — there are a lot of them, one pair for every basic operator. (Are all of them actually needed for the proof?)

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj, a_inv, a_inv_adj, id1, id1_adj, id2, id2_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# id1 is the identity on the source of a, id2 the identity on its target;
# every basic operator has to be multiplied by them from both sides
identity_matrices = [id2*a - a, a*id1 - a,
                     a_dag*id2 - a_dag, id1*a_dag - a_dag,
                     a_inv*id2 - a_inv, id1*a_inv - a_inv,
                     a_adj*id2 - a_adj, id1*a_adj - a_adj,
                     id2*a_dag_adj - a_dag_adj, a_dag_adj*id1 - a_dag_adj,
                     id2*a_inv_adj - a_inv_adj, a_inv_adj*id1 - a_inv_adj]

# encode that a_inv is a two-sided inverse of a -- mind which identity
# matrix appears on which side
a_inverse = ...

assumptions = add_adj(Pinv_a + a_inverse + identity_matrices)
# state the claim A^dag = A^(-1)
claim = ...

proof = certify(assumptions, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 6: The Moore-Penrose inverse via the Gram matrix
*(Fact 18 in [Hog13, Sec. I.5.7])*

It holds that $A^\dagger = (A^\ast A)^\dagger A^\ast$.

*Needed:* an auxiliary indeterminate `b` for the Gram matrix $A^\ast A$, together with its own Moore-Penrose identities. Note that the Gram matrix is Hermitian, so its adjoint is `b` itself — which is what you pass to `pinv` as the third argument.

In [ ]:
F.<a, a_adj, a_dag, a_dag_adj, b, b_adj, b_dag, b_dag_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)
# b_dag is the Moore-Penrose inverse of the Gram matrix b
Pinv_b = pinv(...)
# define b to be the Gram matrix A^* A
b_def = ...

assumptions = add_adj(Pinv_a + Pinv_b + [b_def])
# state the claim A^dag = (A^* A)^dag A^*
claim = ...

proof = certify(assumptions, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 7: The Moore-Penrose inverse of a real matrix is real
*(Fact 2 in [Hog13, Sec. I.5.7])*

If $A$ is a real matrix, then $A^\dagger$ is real as well.

*Needed:* the decomposition of the adjoint into conjugation and transposition. Here `b` denotes $A^\dagger$, and `a_tr`, `a_c` denote $A^T$ and $A^C$. The transposed and conjugated Moore-Penrose identities are given. You have to say that $A$ is real, which amounts to two polynomials: $A = A^C$, and — as a consequence — $A^T = A^\ast$.

In [ ]:
F.<a, a_tr, a_c, a_adj, b, b_tr, b_c, b_adj> = FreeAlgebra(QQ)

# the classical Moore-Penrose identities for b = A^dag, plus their adjoints
Pinv_b = add_adj(pinv(a, b, a_adj, b_adj))
# the transposed identities
Pinv_b_tr = [a_tr*b_tr*a_tr - a_tr, b_tr*a_tr*b_tr - b_tr,
             a_tr*b_tr - b_c*a_c, b_tr*a_tr - a_c*b_c]
# the conjugated identities
Pinv_b_c = [a_c*b_c*a_c - a_c, b_c*a_c*b_c - b_c]

# encode that a is real
a_real = ...

assumptions = Pinv_b + Pinv_b_tr + Pinv_b_c + a_real
# state the claim "A^dag is real"
claim = ...

proof = certify(assumptions, claim)

print("The proof is:")
pretty_print_proof(proof, assumptions)

## Exercise 8: Full rank decomposition
*(Fact 3 in [Hog13, Sec. I.5.7])*

If $A = BC$ is a full rank decomposition, i.e., $B$ has full column rank and $C$ has full row rank, then
$$A^\dagger = C^\ast(B^\ast A C^\ast)^{-1}B^\ast.$$

*Needed:* one-sided inverses and identity matrices. Use `u` for a left inverse of `b`, `v` for a right inverse of `c`, `i` for the identity matrix, and `inv` for the inverse of $B^\ast AC^\ast$.

Note that $B^\ast AC^\ast = B^\ast BCC^\ast$ is indeed invertible, because $B^\ast B$ and $CC^\ast$ are.

In [ ]:
F.<a, b, c, i, u, v, x, a_adj, b_adj, c_adj, i_adj, u_adj, v_adj, x_adj, inv, inv_adj> = FreeAlgebra(QQ)

# x is the Moore-Penrose inverse of a
Pinv_x = pinv(a, x, a_adj, x_adj)
# the decomposition A = BC
a_decomposition = [a - b*c]
# b has full column rank (u is a left inverse of b) and
# c has full row rank (v is a right inverse of c)
full_rank = ...
# inv is a two-sided inverse of b_adj*a*c_adj
inverse = ...
# the identities satisfied by the identity matrix i
identity_matrix = [i*i - i, i - i_adj, b*i - b, i*c - c,
                   i*u - u, v*i - v, inv*i - inv, i*inv - inv]

assumptions = add_adj(Pinv_x + a_decomposition + full_rank + inverse + identity_matrix)
# state the claim A^dag = C^* (B^* A C^*)^(-1) B^*
claim = ...

proof = certify(assumptions, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 9: Projections onto ranges
*(Fact 20 in [Hog13, Sec. I.5.7])*

$AA^\dagger$ is the orthogonal projection onto $\operatorname{range}(A)$. Part of this statement is the equality
$$\operatorname{range}(AA^\dagger) = \operatorname{range}(A),$$
which by Douglas' lemma amounts to two factorisations.

*Needed:* ranges. One of the two inclusions is done for you; find the factorisation that proves the other one.

In [ ]:
F.<a, a_adj, a_dag, a_dag_adj> = FreeAlgebra(QQ)

I = NCIdeal(add_adj(pinv(a, a_dag, a_adj, a_dag_adj)))

# range(A A^dag) is contained in range(A), because A A^dag = A X for some X
print(I.find_equivalent_expression(a*a_dag, prefix=a, heuristic='naive')[0])

# now the other inclusion: range(A) is contained in range(A A^dag)
print(I.find_equivalent_expression(..., prefix=..., heuristic='naive')[0])

## Exercise 10: The reverse order law
*(Fact 25 in [Hog13, Sec. I.5.7])*

In general $(AB)^\dagger \neq B^\dagger A^\dagger$. Several equivalent conditions are known that make the reverse order law hold. One of them is

$$(3) \qquad A^\dagger ABB^\ast A^\ast = BB^\ast A^\ast \quad\text{and}\quad BB^\dagger A^\ast AB = A^\ast AB.$$

Show that condition (3) implies $(AB)^\dagger = B^\dagger A^\dagger$.

*Needed:* Moore-Penrose identities for the *product* $AB$.

In [ ]:
F.<a, b, a_dag, b_dag, ab_dag, a_adj, b_adj, a_dag_adj, b_dag_adj, ab_dag_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)
Pinv_b = pinv(b, b_dag, b_adj, b_dag_adj)
# ab_dag is the Moore-Penrose inverse of the product a*b
Pinv_ab = pinv(...)

basic_assumptions = Pinv_a + Pinv_b + Pinv_ab

# condition (3)
cond_3 = [a_dag*a*b*b_adj*a_adj - b*b_adj*a_adj,
          b*b_dag*a_adj*a*b - a_adj*a*b]
# the reverse order law
rol = [ab_dag - b_dag*a_dag]

# show that condition (3) implies the reverse order law
proofs = certify(...,maxiter=20)
print("\nThe proofs consist of %s steps respectively." % str(list(map(len, proofs))))

# The final test: Hartwig's triple reverse order law

*(Thm. 1 in [Har86], see also Thm. 2.1 in [CIH$^+$21])*

For a product of three factors, several conditions equivalent to
$$(ABC)^\dagger = C^\dagger B^\dagger A^\dagger$$
are known as well. They are all stated in terms of the two auxiliary matrices
$$P = A^\dagger ABCC^\dagger \qquad\text{and}\qquad Q = CC^\dagger B^\dagger A^\dagger A.$$
One of them is
$$(4) \qquad PQP = P, \quad \operatorname{range}(A^\ast A P) = \operatorname{range}(Q^\ast), \quad \operatorname{range}(CC^\ast P^\ast) = \operatorname{range}(Q).$$

Show that condition (4) implies $(ABC)^\dagger = C^\dagger B^\dagger A^\dagger$.

*Needed:* several of the previous ingredients at once — Moore-Penrose identities for $A$, $B$, $C$ and for the product $ABC$, auxiliary indeterminates `p` and `q` for $P$ and $Q$, and Douglas' lemma for the two range equalities. Each equality is an inclusion in both directions, and so gives two factorisations; the auxiliary indeterminates for them are `t`, `u`, `v`, `w`. This is the largest computation in this notebook, and `certify` needs more iterations than usual — but still only a couple of seconds.

In [ ]:
F.<p, p_adj, q, q_adj, a, a_adj, a_dag, a_dag_adj, b, b_adj, b_dag, b_dag_adj, c, c_adj, c_dag, c_dag_adj, abc_dag, abc_dag_adj, t, t_adj, u, u_adj, v, v_adj, w, w_adj> = FreeAlgebra(QQ)

# the Moore-Penrose inverses of a, b, c and of the product a*b*c
...
# the definitions of P and Q
PQ = [p - a_dag*a*b*c*c_dag,
      ...]                     # the analogous definition of q

basic_assumptions = PQ + ...

# condition (4)
cond_4 = [p*q*p - p,
          q_adj*v - a_adj*a*p,   # range(A^* A P) is contained in range(Q^*)
          ...,                   # range(Q^*) is contained in range(A^* A P)
          ...,                   # range(C C^* P^*) is contained in range(Q)
          ...]                   # range(Q) is contained in range(C C^* P^*)

# the triple reverse order law
rol = ...

# show that condition (4) implies the triple reverse order law
proof = certify(..., maxiter=50)
print("\nThe proof consists of %d steps." % len(proof))

# Part 4: When a statement is false

If `certify` cannot verify a claim, it reports

> `Failed! Not all ideal memberships could be verified.`

and returns `False`. This is *not* a proof that the claim is wrong: the search for a proof only runs for a bounded number of iterations, and increasing `maxiter` may well succeed.

To actually refute a statement, the package can search for a counterexample. The command `construct_counterexample` looks for concrete matrices that satisfy all assumptions but violate the claim. By default it searches for square matrices, all of the same prescribed dimension, one for every basic operator.

## An example: weakening Hartwig's condition

Exercise 11 showed that condition (4) implies the triple reverse order law $(ABC)^\dagger = C^\dagger B^\dagger A^\dagger$. In [CIH$^+$21] it was shown that the two range *equalities* in (4) can be weakened to the inclusions
$$\operatorname{range}(A^\ast A P) \subseteq \operatorname{range}(Q^\ast) \quad\text{and}\quad \operatorname{range}(CC^\ast P^\ast) \supseteq \operatorname{range}(Q).$$ 


What about the combination
$$\operatorname{range}(A^\ast A P) \subseteq \operatorname{range}(Q^\ast) \quad\text{and}\quad \operatorname{range}(CC^\ast P^\ast) \subseteq \operatorname{range}(Q)?$$
We first ask `certify`.

In [10]:
F.<p, p_adj, q, q_adj, a, a_adj, a_dag, a_dag_adj, b, b_adj, b_dag, b_dag_adj, c, c_adj, c_dag, c_dag_adj, abc_dag, abc_dag_adj, t, t_adj, v, v_adj> = FreeAlgebra(QQ)

# the Moore-Penrose inverses of a, b, c and of the product a*b*c
Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)
Pinv_b = pinv(b, b_dag, b_adj, b_dag_adj)
Pinv_c = pinv(c, c_dag, c_adj, c_dag_adj)
Pinv_abc = pinv(a*b*c, abc_dag, c_adj*b_adj*a_adj, abc_dag_adj)
# the definitions of P and Q
PQ = [p - a_dag*a*b*c*c_dag, q - c*c_dag*b_dag*a_dag*a]

# P Q P = P, together with the two range inclusions in the same direction
cond = [p*q*p - p,
        q_adj*v - a_adj*a*p,   # range(A^* A P) is contained in range(Q^*)
        q*t - c*c_adj*p_adj]   # range(C C^* P^*) is contained in range(Q)

assumptions = add_adj(PQ + Pinv_a + Pinv_b + Pinv_c + Pinv_abc + cond)
# the triple reverse order law
claim = abc_dag - c_dag*b_dag*a_dag

proof = certify(assumptions, claim, maxiter=40)
print("\ncertify returned %s." % str(proof))

Computing a (partial) Groebner basis and reducing the claims...

Starting iteration 5...
Starting iteration 10...
Starting iteration 15...
Starting iteration 20...
Starting iteration 25...
Starting iteration 30...
Starting iteration 35...
Starting iteration 40...
Failed! Not all ideal memberships could be verified.

certify returned False.


No proof was found. To see whether the statement is actually false, we look for a counterexample. The keyword `dims` gives the common dimension of the matrices to try — here, all matrices are taken to be $2 \times 2$.

In [11]:
counterexample = construct_counterexample(assumptions, claim, dims=2)
print_solution(counterexample, [a, b, c])

a: matrix([
    (0, 0),
    (1, 0)]),
b: matrix([
    (0, 1),
    (1, -1)]),
c: matrix([
    (0, 1),
    (0, 0)]),


So
$$
A = \begin{pmatrix} 0 & 0 \\ 1 & 0 \end{pmatrix}, \qquad
B = \begin{pmatrix} 0 & 1 \\ 1 & -1 \end{pmatrix}, \qquad
C = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}
$$
should be a counterexample. As above, we can check this independently with SageMath's own `pseudoinverse` method. All three matrices are real, so the adjoint is simply the transpose, and a range inclusion $\operatorname{range}(M) \subseteq \operatorname{range}(N)$ can be tested via $NN^\dagger M = M$.

In [8]:
A = matrix(QQ, [[0, 0], [1, 0]])
B = matrix(QQ, [[0, 1], [1, -1]])
C = matrix(QQ, [[0, 1], [0, 0]])

A_dag, B_dag, C_dag = A.pseudoinverse(), B.pseudoinverse(), C.pseudoinverse()
P = A_dag*A*B*C*C_dag
Q = C*C_dag*B_dag*A_dag*A

print("P*Q*P = P?                    %s" % str(P*Q*P == P))
print("range(A^* A P) <= range(Q^*)? %s" % str(A.T*A*P == 0))
print("range(C C^* P^*) <= range(Q)? %s" % str(C*C.T*P.T == 0))

print("\n(A*B*C)^dag =\n%s\n" % str((A*B*C).pseudoinverse()))
print("C^dag*B^dag*A^dag =\n%s\n" % str(C_dag*B_dag*A_dag))

P*Q*P = P?                    True
range(A^* A P) <= range(Q^*)? True
range(C C^* P^*) <= range(Q)? True

(A*B*C)^dag =
[0 0]
[0 0]

C^dag*B^dag*A^dag =
[0 0]
[0 1]



A word of warning: the search is only carried out for the prescribed dimensions. If no counterexample is found, one may still exist in a larger dimension — but the search space grows quickly, so raising the dimensions is not always feasible.

## Try yourself: the reverse order law without extra hypotheses

Exercise 10 showed that condition (3) implies $(AB)^\dagger = B^\dagger A^\dagger$. Convince yourself that some such condition is really needed, i.e., that the reverse order law does *not* hold in general:

1. set up the Moore-Penrose identities for $A$, $B$ and $AB$, and assume nothing else;
2. check that `certify` does not prove $(AB)^\dagger = B^\dagger A^\dagger$;
3. search for a counterexample, taking all three spaces to be $2$-dimensional;
4. verify the counterexample, as above, with SageMath's `pseudoinverse`.

In [ ]:
F.<a, b, a_dag, b_dag, ab_dag, a_adj, b_adj, a_dag_adj, b_dag_adj, ab_dag_adj> = FreeAlgebra(QQ)

# the Moore-Penrose identities for a, b and a*b -- and nothing else
assumptions = ...
# the reverse order law
claim = ...

# 1. certify does not find a proof
proof = certify(assumptions, claim)
print("\ncertify returned %s.\n" % str(proof))

# 2. search for a counterexample, taking all matrices to be 2x2
counterexample = construct_counterexample(...)

In [ ]:
# 3. verify the counterexample with SageMath's own pseudoinverse
A = matrix(QQ, ...)
B = matrix(QQ, ...)

print("(A*B)^dag =\n%s\n" % str((A*B).pseudoinverse()))
print("B^dag*A^dag =\n%s\n" % str(B.pseudoinverse() * A.pseudoinverse()))
print("Is (A*B)^dag = B^dag*A^dag? %s"
      % str((A*B).pseudoinverse() == B.pseudoinverse() * A.pseudoinverse()))

# Where to go from here

Have fun exploring and using the package! Look for other operator statements in the literature or try to (dis)prove your own research statements. If you have suggestions for improvements or you find any bugs, please let us know under [clemens.hofstadler@jku.at](mailto:clemens.hofstadler@jku.at).

# References

For further information on the approach and the underlying theory, see [BHR23]. Most theorems in this notebook are taken from [Hog13]; the triple reverse order law and its variants are discussed in [CIH$^+$21].

[BHR23]&nbsp;&nbsp; Bernauer, K., Hofstadler, C., Regensburger, G. How to Automatise Proofs of Operator Statements: Moore–Penrose Inverse; A Case Study. In: Computer Algebra in Scientific Computing (CASC) 2023. (2023) https://doi.org/10.1007/978-3-031-41724-5_3



[CIH$^+$21]&nbsp;&nbsp;Cvetković-Ilić, D.S., Hofstadler, C., Hossein Poor, J., Milošević, J., Raab, C.G., Regensburger, G. Algebraic proof methods for identities of matrices and operators: improvements of Hartwig’s triple reverse order law. Appl. Math. Comput. 409, 126357 (2021) https://doi.org/10.1016/j.amc.2021.126357



[Har86]&nbsp;&nbsp;Hartwig, R.E. The reverse order law revisited. Linear Algebra Appl. 76, 241–246 (1986)



[Hog13]&nbsp;&nbsp;Hogben L., Handbook of Linear Algebra. CRC press, 2 edn. (2013)